In [ ]:
#STEP 1: Uploading Dataset in Google Colab
#First, I uploaded the Telco Customer Churn dataset into the Google Colab environment to begin the machine learning workflow.
from google.colab import files
uploaded = files.upload()

Saving Telco-Customer-Churn.csv to Telco-Customer-Churn.csv


In [ ]:
#STEP 2: Importing Required Libraries
#I imported all necessary libraries for data preprocessing, visualization, model building, evaluation, and model persistence.
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import joblib

In [ ]:
#STEP 3: Loading the Dataset
#I loaded the dataset using pandas and performed initial inspection to understand its structure and features.
df = pd.read_csv("Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Remove unnecessary column
#I removed the customerID column as it is an identifier and does not contribute to predictive modeling.
df.drop("customerID", axis=1, inplace=True)

# Convert TotalCharges to numeric
#I converted the TotalCharges column into numeric format to ensure proper numerical processing during modeling.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Fill missing values
#I handled missing values using median imputation to maintain data integrity and prevent model errors.
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Convert target column to binary
#I encoded the target variable into binary format to make it compatible with classification algorithms.
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


/tmp/ipython-input-1121050218.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [ ]:
#STEP 5: Splitting Features and Target
X = df.drop("Churn", axis=1)
y = df["Churn"]

#STEP 6: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (5634, 19)
Testing shape: (1409, 19)


In [ ]:
#STEP 7: Identifying Numeric and Categorical Features
# I dynamically identified numerical and categorical features to apply appropriate preprocessing transformations within the pipeline.
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numeric Columns:", numeric_features)
print("Categorical Columns:", categorical_features)

Numeric Columns: Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')
Categorical Columns: Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


In [ ]:
#STEP 8: Creating Numeric Pipeline
# I created a numerical preprocessing pipeline that standardizes numeric features using StandardScaler to ensure consistent
#feature scaling and improved model convergence.
# Numeric pipeline
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

# STEP 9: Creating Categorical Pipeline
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# STEP 10: Combining with ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
#STEP 11: Creating Full Logistic Regression Pipeline
# I constructed an end-to-end machine learning pipeline integrating preprocessing and Logistic Regression into a single reusable workflow.
log_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [ ]:
#STEP 12: Hyperparameter Tuning Using GridSearchCV
# I implemented hyperparameter tuning using GridSearchCV with 5-fold cross-validation
# to optimize the regularization parameter and improve model performance.
param_grid_log = {
    "classifier__C": [0.01, 0.1, 1, 10]
}

grid_log = GridSearchCV(
    log_pipeline,
    param_grid_log,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
#STEP 13: Training the Model
grid_log.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=-1, param_grid={'classifier__C': [0.01, 0.1, 1, 10]},
             scoring='accuracy')

In [ ]:

# STEP 14: Extracting Best Model
best_log_model = grid_log.best_estimator_

# STEP 15: Making Predictions
y_pred_log = best_log_model.predict(X_test)


#STEP 16: Model Evaluation
print("Best Parameters:", grid_log.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Best Parameters: {'classifier__C': 10}
Accuracy: 0.8055358410220014
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [ ]:
# STEP 17: Creating Random Forest Pipeline
# I implemented a second pipeline using Random Forest to compare performance with
#Logistic Regression and evaluate a more complex ensemble-based approach.
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [ ]:
#STEP 18: Defining Hyperparameter Grid
#I defined a hyperparameter search space to optimize model complexity and generalization capability using GridSearchCV.
param_grid_rf = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10],
    "classifier__min_samples_split": [2, 5]
}
#STEP 19: Applying GridSearchCV
#I performed hyperparameter optimization using 5-fold cross-validation to ensure robust model selection and prevent overfitting
grid_rf = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
#STEP 20: Training the Model
grid_rf.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneSe...
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [None, 10],
                         'classifier__min_samples_split': [2, 5],
                         'classifier__n_estimators': [100, 200]},
             scoring='accuracy')

In [ ]:
#STEP 21: Extracting Best Random Forest Model
best_rf_model = grid_rf.best_estimator_


#STEP 22: Making Predictions
y_pred_rf = best_rf_model.predict(X_test)


#STEP 23: Model Evaluation
print("Best Parameters:", grid_rf.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Accuracy: 0.7998580553584103
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [ ]:
#joblib.dump(best_rf_model, "customer_churn_pipeline.pkl")
#I exported the optimized end-to-end pipeline using joblib, enabling model persistence and deployment readiness.
joblib.dump(best_rf_model, "customer_churn_pipeline.pkl")
print("Model Saved Successfully!")

Model Saved Successfully!


In [ ]:
#STEP 25: Downloading the Model File
from google.colab import files
files.download("customer_churn_pipeline.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
